# LangChain RAG — Sample Code (reference template)
### Module 5 sample notebook

A minimal LangChain RAG you can copy into your project, over `ai_framework_kb.md`. Needs `OPENAI_API_KEY` (swap `OpenAIEmbeddings`/`ChatOpenAI` for `HuggingFaceEmbeddings` + a local model to run offline). See `langchain_basics.ipynb` and `langchain_rag.ipynb` for the full lessons.

In [ ]:
# pip install langchain langchain-openai langchain-chroma langchain-community python-dotenv
import os
try:
    from dotenv import load_dotenv, find_dotenv; load_dotenv(find_dotenv())
except Exception:
    pass

if not os.getenv("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY to run this sample (or swap in HuggingFaceEmbeddings + a local model).")
else:
    from langchain_community.document_loaders import TextLoader
    from langchain.text_splitter import CharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_chroma import Chroma
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.runnables import RunnablePassthrough

    # 1) load + split
    docs = TextLoader("ai_framework_kb.md", encoding="utf-8").load()
    chunks = CharacterTextSplitter(chunk_size=900, chunk_overlap=150, separator="\n").split_documents(docs)

    # 2) embed + store
    db = Chroma.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
    retriever = db.as_retriever(search_kwargs={"k": 3})

    # 3) RAG chain (retriever -> prompt -> model -> parser)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using ONLY this context; if unknown, say so.\n\n{context}"),
        ("human", "{question}"),
    ])
    fmt = lambda d: "\n\n".join(x.page_content for x in d)
    chain = ({"context": retriever | fmt, "question": RunnablePassthrough()}
             | prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser())

    print(chain.invoke("What are the four functions of the NIST AI Risk Management Framework?"))